In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time, psutil
import cv2
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from multiprocessing import Pool, cpu_count
import matplotlib.pyplot as plt
from tabulate import tabulate
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
print("CPU Cores:", psutil.cpu_count(logical=True))

DATASET_PATH = "/content/drive/MyDrive/signatures/"
THRESHOLD = 0.8

try:
    dataset_files = [os.path.join(DATASET_PATH, f) for f in os.listdir(DATASET_PATH) if f.endswith(('.jpg', '.jpeg', '.png'))]
    if not dataset_files:
        print("\n⚠️ WARNING: No image files found in the dataset path. Using a dummy list for testing.")
        dataset_files = ["dummy_image_1.png", "dummy_image_2.png"]
except FileNotFoundError:
    print("\n❌ ERROR: Dataset path not found. Please check your drive path.")
    dataset_files = ["dummy_image_1.png"]

print(f"\n✅ Found {len(dataset_files)} signature images for testing.\n")

def create_model():
    return MobileNetV2(weights="imagenet", include_top=False, pooling='avg')
global_model = create_model()

def extract_features(img_path, model_instance=global_model):
    if 'dummy' in img_path:
        time.sleep(0.01)
        return np.zeros(1280)
    try:
        img = image.load_img(img_path, target_size=(224, 224))
        x = image.img_to_array(img)
        x = np.expand_dims(x, axis=0)
        x = preprocess_input(x)
        features = model_instance.predict(x, verbose=0)
        return features.flatten()
    except Exception as e:
        return np.zeros(1280)

global _worker_model
_worker_model = None

def init_worker():
    global _worker_model
    _worker_model = create_model()

def extract_features_multicore(img_path):
    global _worker_model
    if _worker_model is None:
        init_worker()
    return extract_features(img_path, model_instance=_worker_model)

def single_core_features(files):
    return np.array([extract_features(f, model_instance=global_model) for f in files])

def gpu_features(files):
    return np.array([extract_features(f, model_instance=global_model) for f in files])

def measure_performance(mode_name, func, files):
    print(f"\n⏳ Running {mode_name}...")
    start_cpu = psutil.cpu_percent(interval=None)
    start_time = time.time()
    features = func(files)
    end_time = time.time()
    end_cpu = psutil.cpu_percent(interval=None)
    exec_time = end_time - start_time
    cpu_usage = (start_cpu + end_cpu) / 2
    print(f"✅ {mode_name} completed in {exec_time:.2f} sec.")
    return {
        "Mode": mode_name,
        "Images": len(files),
        "Execution Time (s)": round(exec_time, 3),
        "CPU Usage (%)": round(cpu_usage, 2),
        "Feature Shape": features.shape
    }

results = []
results.append(measure_performance("CPU (CPU)", single_core_features, dataset_files))
results.append(measure_performance("GPU (TensorFlow)", gpu_features, dataset_files))

print("\n📊 Performance Comparison Matrix (Lower time is better):\n")
print(tabulate(results, headers="keys", tablefmt="fancy_grid"))

modes = [result["Mode"] for result in results]
execution_times = [result["Execution Time (s)"] for result in results]

plt.figure(figsize=(10, 6))
plt.bar(modes, execution_times, color=['skyblue', 'lightgreen'])
plt.ylabel("Execution Time (s)")
plt.title("Performance Comparison: Execution Time by Mode")
plt.show()

cpu_usages = [result["CPU Usage (%)"] for result in results]

plt.figure(figsize=(10, 6))
plt.bar(modes, cpu_usages, color=['skyblue', 'lightgreen'])
plt.ylabel("CPU Usage (%)")
plt.title("Performance Comparison: CPU Usage by Mode")
plt.show()

try:
    if len(dataset_files) > 1 and not dataset_files[0].startswith("dummy"):
        from google.colab import files
        print("\n\n*** Verification Step: Please upload ONE signature image to test against the dataset. ***")
        uploaded = files.upload()
        uploaded_path = list(uploaded.keys())[0]
    else:
        uploaded_path = dataset_files[0]
except Exception as e:
    print(f"File upload skipped/failed: {e}")
    uploaded_path = dataset_files[0]

def verify_signature(uploaded_path, dataset_features, threshold=THRESHOLD):
    uploaded_feat = extract_features(uploaded_path, model_instance=global_model)
    if np.all(uploaded_feat == 0) and uploaded_path.startswith("dummy"):
         print("\nNOTE: Using dummy features, verification results are meaningless.")
         return "⚠️ Dummy Result (N/A)"
    similarities = cosine_similarity([uploaded_feat], dataset_features)[0]
    best_match = np.max(similarities)
    print("\nBest cosine similarity score:", best_match)
    return "!!! Genuine Signature" if best_match >= threshold else " Forged Signature !!!"

print("\nChoosing GPU feature set for final verification...")
dataset_features = gpu_features(dataset_files)
result = verify_signature(uploaded_path, dataset_features)
print("\nFinal Verification Result:", result)
if not uploaded_path.startswith("dummy"):
    try:
        plt.imshow(cv2.cvtColor(cv2.imread(uploaded_path), cv2.COLOR_BGR2RGB))
        plt.title(result)
        plt.axis('off')
        plt.show()
    except Exception as e:
        print(f"Could not display image: {e}")
